# **BLOCK 1: SETUP, UNZIP DATASET, AND VERIFICATION**

In [1]:
# ============================================================================
# BLOCK 1: SETUP PATHS FOR SHOULDER/ARM MODEL
# ============================================================================

import os
import json
import yaml
import glob
import time

print("="*70)
print("SHOULDER & ARM FRACTURE DETECTION - FASTER R-CNN")
print("="*70)

# ========== YOUR DATASET PATHS ==========
BASE_PATH = "/kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL"

# Paths to your splits
TRAIN_IMAGES = os.path.join(BASE_PATH, "train", "images")
TRAIN_LABELS = os.path.join(BASE_PATH, "train", "labels")
VAL_IMAGES = os.path.join(BASE_PATH, "val", "images")
VAL_LABELS = os.path.join(BASE_PATH, "val", "labels")
TEST_IMAGES = os.path.join(BASE_PATH, "test", "images")
TEST_LABELS = os.path.join(BASE_PATH, "test", "labels")

WORKING_DIR = "/kaggle/working"

print(f"📁 Base path: {BASE_PATH}")
print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Train labels: {TRAIN_LABELS}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Test images: {TEST_IMAGES}")
print(f"📁 Working dir: {WORKING_DIR}")

# Verify paths exist
for path, name in [(TRAIN_IMAGES, "Train images"), (TRAIN_LABELS, "Train labels"),
                   (VAL_IMAGES, "Val images"), (VAL_LABELS, "Val labels"),
                   (TEST_IMAGES, "Test images"), (TEST_LABELS, "Test labels")]:
    exists = os.path.exists(path)
    print(f"  {name}: {'✅' if exists else '❌'} {path if exists else 'Not found'}")

# Count images
train_count = len([f for f in os.listdir(TRAIN_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])
val_count = len([f for f in os.listdir(VAL_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])
test_count = len([f for f in os.listdir(TEST_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])

print(f"\n📊 DATASET STATISTICS:")
print(f"   Training: {train_count} images")
print(f"   Validation: {val_count} images")
print(f"   Test: {test_count} images")
print(f"   TOTAL: {train_count + val_count + test_count} images")

print("="*70)

SHOULDER & ARM FRACTURE DETECTION - FASTER R-CNN
📁 Base path: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL
📁 Train images: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL/train/images
📁 Train labels: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL/train/labels
📁 Val images: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL/val/images
📁 Test images: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL/test/images
📁 Working dir: /kaggle/working
  Train images: ✅ /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-frac


📊 DATASET STATISTICS:
   Training: 12665 images
   Validation: 2235 images
   Test: 549 images
   TOTAL: 15449 images


# **BLOCK 2: INSTALL FASTER R-CNN DEPENDENCIES**

In [2]:
# ============================================================================
# BLOCK 2: INSTALL DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

!pip install -q pandas pyyaml scikit-learn tqdm matplotlib opencv-python
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
from PIL import Image

print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("="*70)

INSTALLING DEPENDENCIES


  Preparing metadata (setup.py) ... done


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/50.2 kB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.3 MB/s eta 0:00:00


  Preparing metadata (setup.py) ... done


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.7 MB/s eta 0:00:00



✅ PyTorch: 2.10.0+cu128
✅ CUDA Available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.6 GB


In [3]:
# Delete old COCO JSON files to force regeneration
import os
json_files = ["/kaggle/working/train_coco.json", "/kaggle/working/val_coco.json", "/kaggle/working/test_coco.json"]
for f in json_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"✅ Deleted: {f}")

# **BLOCK 3: CONVERT YOLO TO COCO FORMAT (FOR FASTER R-CNN)**

In [4]:
# ============================================================================
# BLOCK 3: CONVERT YOLO TO COCO FORMAT (COMPLETE FILTERING)
# ============================================================================

print("="*70)
print("CONVERTING YOLO TO COCO FORMAT")
print("="*70)

import json
from PIL import Image
from tqdm import tqdm
import os

def yolo_to_coco(images_dir, labels_dir, output_json, class_names=['fracture']):
    """
    Convert YOLO format to COCO JSON format
    ONLY includes pairs where BOTH image AND label file exist
    """
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return None
    
    # Get all image files that exist
    existing_images = set([f for f in os.listdir(images_dir) 
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    
    # Get all label files that exist
    existing_labels = set([f for f in os.listdir(labels_dir) 
                           if f.endswith('.txt')])
    
    # Find matching pairs (image exists AND label exists)
    valid_pairs = []
    for img_file in existing_images:
        label_file = os.path.splitext(img_file)[0] + '.txt'
        if label_file in existing_labels:
            valid_pairs.append((img_file, label_file))
    
    print(f"🔄 Found {len(existing_images)} images, {len(existing_labels)} labels")
    print(f"🔄 Valid pairs (image + label): {len(valid_pairs)}")
    
    # COCO categories start at ID 1
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{"id": i+1, "name": name} for i, name in enumerate(class_names)]
    }
    
    annotation_id = 1
    image_id = 1
    skipped_error = 0
    
    for img_file, label_file in tqdm(valid_pairs, desc="Converting"):
        img_path = os.path.join(images_dir, img_file)
        label_path = os.path.join(labels_dir, label_file)
        
        # Get image dimensions
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"⚠️ Could not read {img_file}: {e}")
            skipped_error += 1
            continue
        
        # Add image
        coco_data["images"].append({
            "id": image_id,
            "file_name": img_file,
            "width": width,
            "height": height
        })
        
        # Add annotations
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    # Convert YOLO class 0 → COCO class 1
                    class_id = int(parts[0]) + 1
                    
                    x_center = float(parts[1]) * width
                    y_center = float(parts[2]) * height
                    bbox_width = float(parts[3]) * width
                    bbox_height = float(parts[4]) * height
                    
                    x = x_center - bbox_width/2
                    y = y_center - bbox_height/2
                    
                    coco_data["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": class_id,
                        "bbox": [x, y, bbox_width, bbox_height],
                        "area": bbox_width * bbox_height,
                        "iscrowd": 0
                    })
                    annotation_id += 1
        
        image_id += 1
    
    if skipped_error > 0:
        print(f"⚠️ Skipped {skipped_error} images due to read errors")
    
    # Find orphaned labels (label exists but image doesn't)
    orphaned_labels = []
    for label_file in existing_labels:
        img_file = label_file.replace('.txt', '.jpg')
        if img_file not in existing_images:
            img_file2 = label_file.replace('.txt', '.jpeg')
            if img_file2 not in existing_images:
                orphaned_labels.append(label_file)
    
    if orphaned_labels:
        print(f"⚠️ WARNING: Found {len(orphaned_labels)} label files with no matching image!")
        print(f"   First 5 orphans: {orphaned_labels[:5]}")
    
    with open(output_json, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"✅ Converted {len(coco_data['images'])} images, {len(coco_data['annotations'])} annotations")
    return coco_data

# Convert each split
class_names = ['fracture']

train_json = f"{WORKING_DIR}/train_coco.json"
val_json = f"{WORKING_DIR}/val_coco.json"
test_json = f"{WORKING_DIR}/test_coco.json"

print("\n📊 Converting TRAIN split...")
yolo_to_coco(TRAIN_IMAGES, TRAIN_LABELS, train_json, class_names)

print("\n📊 Converting VALIDATION split...")
yolo_to_coco(VAL_IMAGES, VAL_LABELS, val_json, class_names)

print("\n📊 Converting TEST split...")
yolo_to_coco(TEST_IMAGES, TEST_LABELS, test_json, class_names)

print("\n✅ COCO conversion complete!")
print("="*70)

CONVERTING YOLO TO COCO FORMAT

📊 Converting TRAIN split...


🔄 Found 12665 images, 12665 labels
🔄 Valid pairs (image + label): 12665


Converting:   0%|          | 0/12665 [00:00<?, ?it/s]

Converting:   0%|          | 4/12665 [00:00<05:40, 37.18it/s]

Converting:   0%|          | 9/12665 [00:00<04:44, 44.43it/s]

...

Converting:  97%|█████████▋| 535/549 [00:08<00:00, 64.61it/s]

Converting:  99%|█████████▊| 542/549 [00:08<00:00, 65.24it/s]

Converting: 100%|██████████| 549/549 [00:08<00:00, 63.46it/s]

Converting: 100%|██████████| 549/549 [00:08<00:00, 63.84it/s]

✅ Converted 549 images, 662 annotations

✅ COCO conversion complete!


# **BLOCK 4: REGISTER DATASET WITH DETECTRON2**

In [5]:
# ============================================================================
# BLOCK 4: REGISTER DATASET WITH DETECTRON2
# ============================================================================

print("="*70)
print("REGISTERING DATASET WITH DETECTRON2")
print("="*70)

from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.logger import setup_logger

setup_logger()

# Create unique dataset name with timestamp
timestamp = str(int(time.time()))[-8:]
dataset_name = f"shoulder_arm_{timestamp}"

print(f"📝 Dataset name: {dataset_name}")

# Register datasets
register_coco_instances(f"{dataset_name}_train", {}, train_json, TRAIN_IMAGES)
register_coco_instances(f"{dataset_name}_val", {}, val_json, VAL_IMAGES)
register_coco_instances(f"{dataset_name}_test", {}, test_json, TEST_IMAGES)

# Set metadata (only 1 class: fracture)
class_names = ['fracture']
MetadataCatalog.get(f"{dataset_name}_train").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_val").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_test").thing_classes = class_names

# Verify registration
print("\n📊 Registered datasets:")
for split in ['train', 'val', 'test']:
    name = f"{dataset_name}_{split}"
    if name in DatasetCatalog.list():
        dataset = DatasetCatalog.get(name)
        print(f"  ✅ {name}: {len(dataset)} images")
    else:
        print(f"  ❌ {name} not found!")

# Save dataset name for later
with open(f"{WORKING_DIR}/dataset_name.txt", 'w') as f:
    f.write(dataset_name)

print("\n✅ Dataset registration complete!")
print("="*70)

REGISTERING DATASET WITH DETECTRON2


📝 Dataset name: shoulder_arm_76490551

📊 Registered datasets:


[04/18 05:35:51 d2.data.datasets.coco]: Loaded 12665 images in COCO format from /kaggle/working/train_coco.json


  ✅ shoulder_arm_76490551_train: 12665 images
[04/18 05:35:51 d2.data.datasets.coco]: Loaded 2235 images in COCO format from /kaggle/working/val_coco.json


  ✅ shoulder_arm_76490551_val: 2235 images
[04/18 05:35:51 d2.data.datasets.coco]: Loaded 549 images in COCO format from /kaggle/working/test_coco.json


  ✅ shoulder_arm_76490551_test: 549 images

✅ Dataset registration complete!


# **BLOCK 5: CONFIGURE FASTER R-CNN (ENHANCED)**

In [6]:
# ============================================================================
# BLOCK 5: CONFIGURE FASTER R-CNN (RUN 2: 45,000 to 95,000 iterations)
# ============================================================================

print("="*70)
print("CONFIGURING FASTER R-CNN MODEL (RUN 2 - RESUME FROM 45,000)")
print("="*70)

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
import os
import time
import json

# ========== PATHS ==========
WORKING_DIR = "/kaggle/working"

# Paths to your dataset (the images)
BASE_PATH = "/kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL"

TRAIN_IMAGES = os.path.join(BASE_PATH, "train", "images")
VAL_IMAGES = os.path.join(BASE_PATH, "val", "images")
TEST_IMAGES = os.path.join(BASE_PATH, "test", "images")

# Paths to your COCO JSON files (created by BLOCK 3)
train_json = f"{WORKING_DIR}/train_coco.json"
val_json = f"{WORKING_DIR}/val_coco.json"
test_json = f"{WORKING_DIR}/test_coco.json"

# Paths to your checkpoint files
CHECKPOINT_BASE = "/kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/RUN 1"
CHECKPOINT_WEIGHTS = os.path.join(CHECKPOINT_BASE, "model_0044999.pth")

# Verify JSON files exist
for json_file in [train_json, val_json, test_json]:
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"JSON file not found: {json_file}. Please run BLOCK 3 first!")

print("✅ COCO JSON files found!")

# ========== STEP 2: REGISTER DATASET ==========
print("\n" + "="*70)
print("REGISTERING DATASET")
print("="*70)

# Create a new unique dataset name for this run
dataset_name = f"shoulder_arm_{int(time.time())}"
print(f"📝 New dataset name: {dataset_name}")

# Register datasets using the JSON files from BLOCK 3
register_coco_instances(f"{dataset_name}_train", {}, train_json, TRAIN_IMAGES)
register_coco_instances(f"{dataset_name}_val", {}, val_json, VAL_IMAGES)
register_coco_instances(f"{dataset_name}_test", {}, test_json, TEST_IMAGES)

# Set metadata
class_names = ['fracture']
MetadataCatalog.get(f"{dataset_name}_train").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_val").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_test").thing_classes = class_names

# Save dataset name for later
with open(f"{WORKING_DIR}/dataset_name.txt", 'w') as f:
    f.write(dataset_name)

print(f"✅ Dataset registered as: {dataset_name}")

# ========== STEP 3: CREATE CONFIGURATION ==========
print("\n" + "="*70)
print("CREATING CONFIGURATION")
print("="*70)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# Dataset
cfg.DATASETS.TRAIN = (f"{dataset_name}_train",)
cfg.DATASETS.TEST = (f"{dataset_name}_val",)

# Classes (only fracture)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# ========== RUN 2 TRAINING PARAMETERS (45,000 → 95,000) ==========
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.000125
cfg.SOLVER.MAX_ITER = 95000  # Target: 45,000 → 95,000
cfg.SOLVER.STEPS = (120000, 145000)
cfg.SOLVER.WEIGHT_DECAY = 0.0005
cfg.SOLVER.CHECKPOINT_PERIOD = 7500

# Enhancements
cfg.SOLVER.WARMUP_ITERS = 2500
cfg.SOLVER.WARMUP_FACTOR = 0.001

# Data augmentation
cfg.INPUT.RANDOM_FLIP = "horizontal"

# Model settings
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.5
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32, 64, 128, 256, 512]]

# Input size
cfg.INPUT.MIN_SIZE_TRAIN = (800,)
cfg.INPUT.MAX_SIZE_TRAIN = 800
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 800

# ========== LOAD THE CHECKPOINT WEIGHTS ==========
cfg.MODEL.WEIGHTS = CHECKPOINT_WEIGHTS

# ========== OUTPUT DIRECTORY ==========
cfg.OUTPUT_DIR = f"{WORKING_DIR}/shoulder_arm_model_35epochs_RUN2"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(f"\n⚙️ RUN 2 CONFIGURATION (45,000 → 95,000 iterations):")
print(f"  🏗️ Model: Faster R-CNN with ResNet-50 FPN")
print(f"  📋 Classes: {cfg.MODEL.ROI_HEADS.NUM_CLASSES}")
print(f"  📦 Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"  🔄 Starting from checkpoint: {CHECKPOINT_WEIGHTS}")
print(f"  🎯 Target MAX_ITER: {cfg.SOLVER.MAX_ITER:,}")
print(f"  📊 New iterations to run: {cfg.SOLVER.MAX_ITER - 45000:,}")
print(f"  📉 Learning rate: {cfg.SOLVER.BASE_LR}")
print(f"  💾 Checkpoint period: {cfg.SOLVER.CHECKPOINT_PERIOD}")
print(f"  ⏱️ Expected training time: 10-11 hours")
print(f"  📁 Output: {cfg.OUTPUT_DIR}")

# Save config
with open(f"{cfg.OUTPUT_DIR}/config.yaml", 'w') as f:
    f.write(cfg.dump())

print("\n✅ Configuration saved!")
print("="*70)

CONFIGURING FASTER R-CNN MODEL (RUN 2 - RESUME FROM 45,000)


✅ COCO JSON files found!

REGISTERING DATASET
📝 New dataset name: shoulder_arm_1776490552
✅ Dataset registered as: shoulder_arm_1776490552

CREATING CONFIGURATION

⚙️ RUN 2 CONFIGURATION (45,000 → 95,000 iterations):
  🏗️ Model: Faster R-CNN with ResNet-50 FPN
  📋 Classes: 1
  📦 Batch size: 4
  🔄 Starting from checkpoint: /kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/RUN 1/model_0044999.pth
  🎯 Target MAX_ITER: 95,000
  📊 New iterations to run: 50,000
  📉 Learning rate: 0.000125
  💾 Checkpoint period: 7500
  ⏱️ Expected training time: 10-11 hours
  📁 Output: /kaggle/working/shoulder_arm_model_35epochs_RUN2

✅ Configuration saved!


# **BLOCK 6: TRAIN FASTER R-CNN (ENHANCED WITH EARLY STOPPING)**

In [7]:
# ============================================================================
# BLOCK 6: TRAIN FASTER R-CNN MODEL (WITH AP50 TRACKING & RESUME SUPPORT)
# ============================================================================

print("="*70)
print("TRAINING FASTER R-CNN MODEL")
print("="*70)

import glob
import re
import json
import csv
import time
import torch
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.checkpoint import DetectionCheckpointer

# ============================================================================
# ENHANCED TRAINER WITH AP50 TRACKING
# ============================================================================

class EnhancedTrainer(DefaultTrainer):
    
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)
    
    def __init__(self, cfg):
        super().__init__(cfg)
        self.best_val_ap = 0.0
        self.patience_counter = 0
        self.patience = 5
        self.dataset_name = None
        self.ap50_history = []

        # Load dataset name
        dataset_name_file = os.path.join(WORKING_DIR, "dataset_name.txt")
        if os.path.exists(dataset_name_file):
            with open(dataset_name_file, 'r') as f:
                self.dataset_name = f.read().strip()
        else:
            alt_file = os.path.join(cfg.OUTPUT_DIR, "dataset_name.txt")
            if os.path.exists(alt_file):
                with open(alt_file, 'r') as f:
                    self.dataset_name = f.read().strip()
            else:
                self.dataset_name = "shoulder_arm_dataset"

        # Load existing AP50 history if resuming
        history_file = os.path.join(cfg.OUTPUT_DIR, "ap50_history.json")
        if os.path.exists(history_file):
            with open(history_file, 'r') as f:
                self.ap50_history = json.load(f)
            print(f"✅ Loaded {len(self.ap50_history)} previous AP50 records")

    def after_step(self):
        super().after_step()
        if self.iter % self.cfg.SOLVER.CHECKPOINT_PERIOD == 0 and self.iter > 0:
            self.evaluate_and_check_early_stop()

    def evaluate_and_check_early_stop(self):
        from detectron2.evaluation import inference_on_dataset
        from detectron2.data import build_detection_test_loader

        print(f"\n{'='*50}")
        print(f"📊 EVALUATING AT ITERATION {self.iter}")
        print(f"{'='*50}")

        evaluator = COCOEvaluator(
            f"{self.dataset_name}_val",
            self.cfg,
            False,
            output_dir=self.cfg.OUTPUT_DIR
        )
        val_loader = build_detection_test_loader(self.cfg, f"{self.dataset_name}_val")
        results = inference_on_dataset(self.model, val_loader, evaluator)

        current_ap = results['bbox']['AP50']
        current_time = time.strftime('%Y-%m-%d %H:%M:%S')

        print(f"\n   📈 Current AP50: {current_ap:.2f}%")
        print(f"   🕐 Time: {current_time}")

        self.ap50_history.append({
            'iteration': self.iter,
            'ap50': current_ap,
            'timestamp': current_time
        })

        # Save history to JSON
        history_file = os.path.join(self.cfg.OUTPUT_DIR, "ap50_history.json")
        with open(history_file, 'w') as f:
            json.dump(self.ap50_history, f, indent=2)
        print(f"   💾 AP50 history saved to {history_file}")

        # Save to CSV
        csv_path = os.path.join(self.cfg.OUTPUT_DIR, "ap50_progress.csv")
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['iteration', 'ap50', 'timestamp'])
            for record in self.ap50_history:
                writer.writerow([record['iteration'], record['ap50'], record['timestamp']])
        print(f"   💾 AP50 progress saved to {csv_path}")

        # Save best model
        if current_ap > self.best_val_ap:
            self.best_val_ap = current_ap
            self.patience_counter = 0
            DetectionCheckpointer(self.model).save("best_model")
            print(f"\n   🏆 NEW BEST MODEL! AP50: {current_ap:.2f}%")
        else:
            self.patience_counter += 1
            print(f"\n   📉 No improvement. Patience: {self.patience_counter}/{self.patience}")
            print(f"   Best AP50 so far: {self.best_val_ap:.2f}%")

        # Early stopping
        if self.patience_counter >= self.patience and self.iter > 10000:
            print(f"\n{'='*50}")
            print(f"🛑 EARLY STOPPING TRIGGERED!")
            print(f"   No improvement for {self.patience} evaluations.")
            print(f"   Best AP50: {self.best_val_ap:.2f}%")
            print(f"{'='*50}")
            self._trainer.stop()

# ============================================================================
# INITIALIZE TRAINER
# ============================================================================

trainer = EnhancedTrainer(cfg)

# Load RUN1 checkpoint weights
DetectionCheckpointer(trainer.model).load(cfg.MODEL.WEIGHTS)
print(f"✅ Loaded weights from: {cfg.MODEL.WEIGHTS}")

# Set BOTH iter and start_iter so the LR scheduler is correct
trainer.start_iter = 44999
trainer.iter = 44999
print(f"📊 Starting from iteration: {trainer.start_iter}")

# ============================================================================
# GPU INFORMATION
# ============================================================================

print(f"\n{'='*50}")
print("🎮 GPU INFORMATION")
print(f"{'='*50}")
print(f"   GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"   Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

# ============================================================================
# TRAINING INFORMATION
# ============================================================================

start_time = time.time()

print(f"\n{'='*50}")
print("🔥 TRAINING INFORMATION")
print(f"{'='*50}")
print(f"   Training resumed at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Current Iteration: {trainer.start_iter}")
print(f"   Target MAX_ITER: {cfg.SOLVER.MAX_ITER:,}")
print(f"   Training for: {cfg.SOLVER.MAX_ITER - trainer.start_iter:,} more iterations")
print(f"   Checkpoint period: {cfg.SOLVER.CHECKPOINT_PERIOD} iterations")
print(f"   Total validations in this run: {(cfg.SOLVER.MAX_ITER - trainer.start_iter) // cfg.SOLVER.CHECKPOINT_PERIOD} times")
print(f"{'='*50}")

print(f"\n💡 ENHANCED FEATURES:")
print("   ✅ Resume from iteration 44999")
print("   ✅ Early stopping (patience=5)")
print("   ✅ Best model saving")
print("   ✅ AP50 tracking (saved to JSON + CSV)")
print("   ✅ Learning rate warmup")

# ============================================================================
# START TRAINING
# ============================================================================

try:
    trainer.train()

    duration = (time.time() - start_time) / 3600

    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"⏱️ Training time: {duration:.2f} hours")
    print(f"💾 Final model: {cfg.OUTPUT_DIR}/model_final.pth")
    print(f"🏆 Best model: {cfg.OUTPUT_DIR}/best_model.pth (AP50: {trainer.best_val_ap:.2f}%)")
    print(f"📊 AP50 history: {cfg.OUTPUT_DIR}/ap50_history.json")
    print(f"📊 AP50 progress CSV: {cfg.OUTPUT_DIR}/ap50_progress.csv")

    # Print AP50 summary
    print("\n" + "="*70)
    print("📊 AP50 PROGRESS SUMMARY")
    print("="*70)
    print(f"{'Iteration':>12} | {'AP50 (%)':>10} | {'Improvement':>12}")
    print("-" * 50)

    prev_ap = None
    for record in trainer.ap50_history:
        if prev_ap is not None:
            improvement = record['ap50'] - prev_ap
            imp_str = f"+{improvement:.2f}%" if improvement > 0 else f"{improvement:.2f}%"
        else:
            imp_str = "N/A"
        print(f"{record['iteration']:>12,} | {record['ap50']:>10.2f} | {imp_str:>12}")
        prev_ap = record['ap50']
    print("="*70)

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted - checkpoints saved")
    print(f"💾 Last checkpoint: {cfg.OUTPUT_DIR}/model_{trainer.iter}.pth")
except Exception as e:
    print(f"\n❌ Error: {e}")
    emergency_path = os.path.join(cfg.OUTPUT_DIR, f"emergency_iter_{trainer.iter}.pth")
    DetectionCheckpointer(trainer.model).save(emergency_path)
    print(f"🆘 Emergency checkpoint saved to {emergency_path}")

print("="*70)

TRAINING FASTER R-CNN MODEL


[04/18 05:35:53 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

[04/18 05:35:53 d2.data.datasets.coco]: Loaded 12665 images in COCO format from /kaggle/working/train_coco.json


[04/18 05:35:53 d2.data.build]: Removed 0 images with no usable annotations. 12665 images left.


[04/18 05:35:54 d2.data.build]: Distribution of instances among all 1 categories:
|  category  | #instances   |
|:----------:|:-------------|
|  fracture  | 14387        |
|            |              |


...

[04/18 17:27:37 d2.utils.events]:  eta: 0:48:25  iter: 91719  total_loss: 0.542  loss_cls: 0.1235  loss_box_reg: 0.2218  loss_rpn_cls: 0.04006  loss_rpn_loc: 0.1222    time: 0.8823  last_time: 0.8818  data_time: 0.0126  last_data_time: 0.0108   lr: 0.000125  max_mem: 3074M


[04/18 17:27:54 d2.utils.events]:  eta: 0:48:08  iter: 91739  total_loss: 0.52  loss_cls: 0.1074  loss_box_reg: 0.195  loss_rpn_cls: 0.03665  loss_rpn_loc: 0.1255    time: 0.8823  last_time: 0.8905  data_time: 0.0137  last_data_time: 0.0248   lr: 0.000125  max_mem: 3074M


[04/18 17:28:12 d2.utils.events]:  eta: 0:47:50  iter: 91759  total_loss: 0.5213  loss_cls: 0.1059  loss_box_reg: 0.2135  loss_rpn_cls: 0.03729  loss_rpn_loc: 0.1209    time: 0.8823  last_time: 0.8902  data_time: 0.0134  last_data_time: 0.0111   lr: 0.000125  max_mem: 3074M


[04/18 17:28:30 d2.utils.events]:  eta: 0:47:32  iter: 91779  total_loss: 0.493  loss_cls: 0.109  loss_box_reg: 0.2141  loss_rpn_cls: 0.03501  loss_rpn_loc: 0.1326    time: 0.8823  last_time: 0.8758  data_time: 0.0112  last_data_time: 0.0110   lr: 0.000125  max_mem: 3074M


[04/18 17:28:47 d2.utils.events]:  eta: 0:47:14  iter: 91799  total_loss: 0.4896  loss_cls: 0.1141  loss_box_reg: 0.2049  loss_rpn_cls: 0.0335  loss_rpn_loc: 0.1311    time: 0.8823  last_time: 0.8802  data_time: 0.0139  last_data_time: 0.0099   lr: 0.000125  max_mem: 3074M


# **BLOCK 6.5: EXTRACT AND SAVE ALL AP50 VALUES FROM TRAINING**

In [ ]:
# ============================================================================
# BLOCK 6.5: EXTRACT AND SAVE ALL AP50 VALUES FROM TRAINING
# ============================================================================

print("="*70)
print("EXTRACTING AP50 VALUES FROM TRAINING LOGS")
print("="*70)

import re
import json
from pathlib import Path

# Path to your output directory
output_dir = cfg.OUTPUT_DIR

# Find all evaluation lines in the logs
# Detectron2 prints: "Current AP50: XX.XX%"
log_file = None

# Try to find the log file
possible_logs = list(Path(output_dir).glob("*.log")) + \
                list(Path("/kaggle/working").glob("*.log")) + \
                list(Path("/kaggle/working/.virtual_documents").glob("*.log"))

ap50_records = []

# If you have the notebook output saved as text, you can parse it
# For now, we'll create a manual tracking file during training

# Method 1: If you saved metrics.json during training
metrics_file = os.path.join(output_dir, "metrics.json")
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    print(f"✅ Found metrics.json with {len(metrics)} entries")
    
    # Convert to list of (iteration, ap50) if metrics is dict
    if isinstance(metrics, dict):
        for iter_str, ap50 in metrics.items():
            ap50_records.append((int(iter_str), ap50))
    elif isinstance(metrics, list):
        ap50_records = metrics

# Method 2: Create tracking file during training (add this to EnhancedTrainer)
# We'll modify the trainer to save AP50 after each evaluation

# Sort by iteration
ap50_records.sort(key=lambda x: x[0])

print(f"\n📊 AP50 PROGRESS:")
print("-" * 40)
for iteration, ap50 in ap50_records:
    print(f"   Iteration {iteration:,}: AP50 = {ap50:.2f}%")
print("-" * 40)

# Save to CSV for easy viewing
import csv
csv_path = os.path.join(output_dir, "ap50_progress.csv")
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['iteration', 'ap50'])
    for iteration, ap50 in ap50_records:
        writer.writerow([iteration, ap50])

print(f"\n✅ AP50 progress saved to: {csv_path}")

# Store for PDF generation
ap50_data = ap50_records

# **BLOCK 7: EVALUATE ON TEST SET**

In [ ]:
# ============================================================================
# BLOCK 7: EVALUATE ON TEST SET (FIXED)
# ============================================================================

print("="*70)
print("EVALUATING MODEL ON TEST SET")
print("="*70)

from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# Load best model
best_model = os.path.join(cfg.OUTPUT_DIR, "best_model.pth")
if os.path.exists(best_model):
    cfg.MODEL.WEIGHTS = best_model
    print(f"✅ Using best model: {best_model}")
else:
    cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    print(f"✅ Using final model: {cfg.MODEL.WEIGHTS}")

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
predictor = DefaultPredictor(cfg)

# Evaluate on test set
evaluator = COCOEvaluator(f"{dataset_name}_test", cfg, False, output_dir=cfg.OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, f"{dataset_name}_test")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

print("\n" + "="*70)
print("📊 TEST RESULTS")
print("="*70)

# Store results safely (handles different metric names)
bbox_results = results['bbox']
test_results = {
    'AP': bbox_results.get('AP', 0),
    'AP50': bbox_results.get('AP50', 0),
    'AP75': bbox_results.get('AP75', 0),
    'AR1': bbox_results.get('AR@1', bbox_results.get('AR1', 0)),
    'AR10': bbox_results.get('AR@10', bbox_results.get('AR10', 0)),
    'AR100': bbox_results.get('AR@100', bbox_results.get('AR100', 0))
}

print(f"\n🎯 Average Precision (AP):")
print(f"   AP @ IoU=0.50:0.95: {test_results['AP']:.2f}%")
print(f"   AP50 @ IoU=0.50: {test_results['AP50']:.2f}%")
print(f"   AP75 @ IoU=0.75: {test_results['AP75']:.2f}%")

print(f"\n📈 Average Recall (AR):")
print(f"   AR @ 1 detection: {test_results['AR1']:.2f}%")
print(f"   AR @ 10 detections: {test_results['AR10']:.2f}%")
print(f"   AR @ 100 detections: {test_results['AR100']:.2f}%")

# Save results
with open(f"{cfg.OUTPUT_DIR}/test_results.json", 'w') as f:
    json.dump(results, f, indent=2)

with open(f"{cfg.OUTPUT_DIR}/metrics.json", 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n✅ Results saved to {cfg.OUTPUT_DIR}/test_results.json")
print("="*70)

# **BLOCK 8: GENERATE FORMAL PDF REPORT**

In [ ]:
# ============================================================================
# BLOCK 8: GENERATE RUN-SPECIFIC PDF REPORT
# ============================================================================

print("="*70)
print(f"GENERATING PDF REPORT FOR RUN {RUN_NUMBER}")
print("="*70)

!pip install -q fpdf

from fpdf import FPDF
import datetime
import json
import csv

# ========== CONFIGURATION ==========
RUN_NUMBER = 1  # CHANGE THIS FOR EACH RUN: 1, 2, 3, or 4
RUN_DESCRIPTION = {
    1: "Initial Training (0 → 54,000 iterations)",
    2: "Resume Training (54,000 → 108,000 iterations)",
    3: "Resume Training (108,000 → 162,000 iterations)",
    4: "Final Training (162,000 → 166,000 iterations)"
}

# Load AP50 history
ap50_history_file = os.path.join(cfg.OUTPUT_DIR, "ap50_history.json")
ap50_data = []
if os.path.exists(ap50_history_file):
    with open(ap50_history_file, 'r') as f:
        ap50_data = json.load(f)
else:
    # Try CSV
    csv_file = os.path.join(cfg.OUTPUT_DIR, "ap50_progress.csv")
    if os.path.exists(csv_file):
        with open(csv_file, 'r') as f:
            reader = csv.DictReader(f)
            ap50_data = [{'iteration': int(r['iteration']), 'ap50': float(r['ap50']), 'timestamp': r['timestamp']} for r in reader]

# Load metrics
metrics_file = os.path.join(cfg.OUTPUT_DIR, "metrics.json")
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)

# Get best AP50
best_ap50 = max([r['ap50'] for r in ap50_data]) if ap50_data else 0
final_ap50 = ap50_data[-1]['ap50'] if ap50_data else 0

# Create PDF
class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 16)
        self.cell(0, 10, f'AETHEA Bone Fracture Detection - Run {RUN_NUMBER}', 0, 1, 'C')
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, RUN_DESCRIPTION.get(RUN_NUMBER, "Training Run"), 0, 1, 'C')
        self.ln(5)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()} | Run {RUN_NUMBER}', 0, 0, 'C')

pdf = PDF()
pdf.add_page()
pdf.set_font('Arial', '', 11)

# Date
date_str = datetime.datetime.now().strftime("%B %d, %Y")
pdf.cell(0, 10, f"Report Date: {date_str}", 0, 1)
pdf.cell(0, 10, f"Run Type: {RUN_DESCRIPTION.get(RUN_NUMBER, 'Training Run')}", 0, 1)
pdf.ln(5)

# Summary Statistics
pdf.set_font('Arial', 'B', 14)
pdf.cell(0, 10, '1. Run Summary', 0, 1)
pdf.set_font('Arial', '', 11)

# Create summary table
pdf.set_font('Arial', 'B', 11)
pdf.cell(70, 8, 'Metric', 1, 0, 'C')
pdf.cell(70, 8, 'Value', 1, 1, 'C')
pdf.set_font('Arial', '', 11)

start_iter = ap50_data[0]['iteration'] if ap50_data else 0
end_iter = ap50_data[-1]['iteration'] if ap50_data else cfg.SOLVER.MAX_ITER

pdf.cell(70, 8, 'Start Iteration', 1, 0)
pdf.cell(70, 8, f'{start_iter:,}', 1, 1)
pdf.cell(70, 8, 'End Iteration', 1, 0)
pdf.cell(70, 8, f'{end_iter:,}', 1, 1)
pdf.cell(70, 8, 'Total Validations', 1, 0)
pdf.cell(70, 8, f'{len(ap50_data)}', 1, 1)
pdf.cell(70, 8, 'Best AP50 Achieved', 1, 0)
pdf.cell(70, 8, f'{best_ap50:.2f}%', 1, 1)
pdf.cell(70, 8, 'Final AP50', 1, 0)
pdf.cell(70, 8, f'{final_ap50:.2f}%', 1, 1)
pdf.ln(5)

# AP50 Progress Table
pdf.set_font('Arial', 'B', 14)
pdf.cell(0, 10, '2. AP50 Progress During Run', 0, 1)
pdf.set_font('Arial', 'B', 10)

# Table headers
pdf.cell(50, 8, 'Iteration', 1, 0, 'C')
pdf.cell(50, 8, 'AP50 (%)', 1, 0, 'C')
pdf.cell(60, 8, 'Timestamp', 1, 1, 'C')

pdf.set_font('Arial', '', 9)
for record in ap50_data:
    pdf.cell(50, 7, f"{record['iteration']:,}", 1, 0, 'C')
    pdf.cell(50, 7, f"{record['ap50']:.2f}%", 1, 0, 'C')
    pdf.cell(60, 7, record.get('timestamp', 'N/A')[:16], 1, 1, 'C')
pdf.ln(5)

# Next Steps
pdf.set_font('Arial', 'B', 14)
pdf.cell(0, 10, '3. Next Steps', 0, 1)
pdf.set_font('Arial', '', 11)

if RUN_NUMBER < 4:
    pdf.multi_cell(0, 6, f"This run completed {end_iter - start_iter:,} iterations. "
                         f"The next run should start from iteration {end_iter:,} "
                         f"and target {end_iter + 54000:,} iterations (or 166,000 total).")
else:
    pdf.multi_cell(0, 6, f"Training complete! Final model achieved {final_ap50:.2f}% AP50. "
                         f"The model is ready for deployment or further fine-tuning.")

# Footer
pdf.ln(10)
pdf.set_font('Arial', 'I', 9)
pdf.cell(0, 10, f"Report Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", 0, 1)
pdf.cell(0, 10, f"Output Directory: {cfg.OUTPUT_DIR}", 0, 1)

# Save PDF
pdf_path = f"{cfg.OUTPUT_DIR}/RUN_{RUN_NUMBER}_REPORT.pdf"
pdf.output(pdf_path)

print(f"✅ PDF report generated: {pdf_path}")

# Display download link
from IPython.display import FileLink, display
display(FileLink(pdf_path))

print(f"\n📊 AP50 Summary:")
for record in ap50_data:
    print(f"   Iter {record['iteration']:,}: {record['ap50']:.2f}%")

In [ ]:
# ============================================================================
# COPY MODEL TO A NEW KAGGLE DATASET
# ============================================================================

import os
import shutil

# Create a temporary folder
temp_dir = "/kaggle/working/model_export"
os.makedirs(temp_dir, exist_ok=True)

# Copy model and config
shutil.copy("/kaggle/working/shoulder_arm_model/model_final.pth", temp_dir)
shutil.copy("/kaggle/working/shoulder_arm_model/config.yaml", temp_dir)

# Create metadata
metadata = {
    "title": "shoulder-arm-fracture-model",
    "id": "andrewwageh/shoulder-arm-model",
    "licenses": [{"name": "CC0-1.0"}]
}

import json
with open(f"{temp_dir}/dataset-metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

# Create zip
!zip -r /kaggle/working/model_dataset.zip /kaggle/working/model_export/

print("✅ Model packaged. Download this zip:")
from IPython.display import FileLink
display(FileLink("/kaggle/working/model_dataset.zip"))

In [ ]:
import os
from IPython.display import HTML

zip_path = "/kaggle/working/model_dataset.zip"
file_size = os.path.getsize(zip_path) / (1024*1024)

# Create HTML download button
html = f'''
<a href="{zip_path}" download="model_dataset.zip" 
   style="display: inline-block; padding: 10px 20px; background-color: #4CAF50; 
          color: white; text-decoration: none; border-radius: 5px; font-size: 16px;">
   📥 Download Model Zip ({file_size:.1f} MB)
</a>
'''
display(HTML(html))